In [ ]:
import pandas as pd

from TerraFin import configure, load_terrafin_config
from TerraFin.analytics.analysis.fundamental import build_sp500_dcf_payload, build_stock_dcf_payload


configure()
config = load_terrafin_config()

### S&P 500 Index DCF Payload

In [ ]:
sp500_payload = build_sp500_dcf_payload()

print(sp500_payload)

pd.Series(
    {
        "status": sp500_payload["status"],
        "symbol": sp500_payload["symbol"],
        "asOf": sp500_payload["asOf"],
        "currentPrice": sp500_payload["currentPrice"],
        "currentIntrinsicValue": sp500_payload["currentIntrinsicValue"],
        "upsidePct": sp500_payload["upsidePct"],
        "rateSource": sp500_payload["rateCurve"]["source"],
    }
)

In [ ]:
pd.DataFrame(sp500_payload.get("methods") or [])[
    ["label", "weight", "currentIntrinsicValue", "upsidePct", "description"]
]

In [ ]:
pd.DataFrame.from_dict(sp500_payload["scenarios"], orient="index")[
    ["label", "intrinsicValue", "upsidePct", "terminalGrowthPct", "terminalDiscountRatePct"]
]

In [ ]:
pd.DataFrame(sp500_payload["scenarios"]["base"]["projectedCashFlows"])[
    ["forecastDate", "growthPct", "cashFlowPerShare", "discountRatePct", "presentValue"]
]

In [ ]:
sensitivity = pd.DataFrame(sp500_payload["sensitivity"]["cells"])
sensitivity.pivot(
    index="terminalGrowthShiftBps",
    columns="discountRateShiftBps",
    values="intrinsicValue",
).sort_index(ascending=False)

### Stock DCF Payload

In [ ]:
stock_payload = build_stock_dcf_payload("NVDA")

pd.Series(
    {
        "status": stock_payload["status"],
        "symbol": stock_payload["symbol"],
        "currentPrice": stock_payload["currentPrice"],
        "currentIntrinsicValue": stock_payload["currentIntrinsicValue"],
        "upsidePct": stock_payload["upsidePct"],
        "dataQuality": stock_payload["dataQuality"]["mode"],
        "warningCount": len(stock_payload["warnings"]),
    }
)

In [ ]:
if stock_payload["scenarios"]:
    pd.DataFrame.from_dict(stock_payload["scenarios"], orient="index")[
        ["label", "intrinsicValue", "upsidePct", "terminalGrowthPct", "terminalDiscountRatePct"]
    ]
else:
    pd.Series({"status": stock_payload["status"], "warnings": stock_payload["warnings"]})